# Exercício 1 — Índice Invertido com Apache Spark

Neste exercício vais construir o mesmo **índice invertido** do exercício MapReduce,
desta vez usando PySpark. O objectivo é duplo: perceber como o Spark funciona,
e ver como o mesmo problema se expressa de forma mais directa.

---

### O que muda face ao MapReduce?

No exercício Hadoop tinhas dois ficheiros separados que comunicavam por `stdin`/`stdout`:

```
mapper.py   →   stdout   →   [Hadoop Shuffle & Sort]   →   stdin   →   reducer.py
```

Em Spark, toda a lógica fica num único programa. As operações são encadeadas
sobre um **RDD** — uma colecção distribuída que parece uma lista Python mas que
o Spark processa em paralelo nos nós disponíveis:

```
parsed  →  .flatMap()  →  .groupByKey()  →  .mapValues()
             (mapper)       (shuffle)         (reducer)
```

| Hadoop Streaming | PySpark |
|---|---|
| `mapper.py` (script separado) | `.flatMap(lambda doc: ...)` |
| Hadoop Shuffle & Sort | `.groupByKey()` |
| `reducer.py` (script separado) | `.mapValues(lambda doc_ids: ...)` |
| Dados intermédios em disco | Dados intermédios em memória |
| Um ficheiro de resultado | Um RDD em memória |

### Lazy evaluation

Quando escreves `.flatMap().groupByKey().mapValues()`, o Spark **não executa nada ainda**.
Está apenas a construir um plano — o DAG de transformações. Só quando chamares uma
**acção** (como `.take()`, `.collect()` ou `.saveAsTextFile()`) é que o Spark executa
tudo de uma vez, de forma optimizada.

Podes ver o DAG em tempo real em **http://localhost:4040** durante a execução.

---
## 1. Setup

In [ ]:
from pyspark import SparkContext, SparkConf
import re

conf = SparkConf().setAppName("IR-Indice-Invertido").setMaster("local[*]")
sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

print(f"Spark versão : {sc.version}")
print(f"Modo         : {sc.master}")
print(f"Spark UI     : http://localhost:4040")

`local[*]` significa modo local com todos os cores disponíveis.
Não precisa de cluster — o Spark simula a distribuição com threads na mesma máquina.

---
## 2. Carregar a Coleção

In [ ]:
# sc.textFile() NÃO lê o ficheiro agora — cria apenas um RDD lazy
docs = sc.textFile("colecao_grande.txt")

# .count() é uma acção — aqui é que a leitura acontece de facto
print(f"Total de documentos: {docs.count()}")
print("\nPrimeiros 3 documentos:")
for linha in docs.take(3):
    print(" ", linha[:110] + "..." if len(linha) > 110 else linha)

In [ ]:
# Cada linha tem o formato:  doc_id<TAB>texto
# Esta função de parsing substitui a leitura do stdin no mapper.py
def parse_doc(linha):
    partes = linha.strip().split("\t", 1)
    return (partes[0], partes[1]) if len(partes) == 2 else None

parsed = docs.map(parse_doc).filter(lambda x: x is not None)

# Verificar o parsing
doc_id, texto = parsed.first()
print(f"doc_id : {doc_id}")
print(f"texto  : {texto[:100]}...")

---
## 3. Construir o Índice Invertido

Vamos construir o índice em três passos explícitos para que a correspondência
com o exercício MapReduce fique clara.

In [ ]:
# Passo 1 — equivalente ao mapper.py
# Para cada documento, emite um par (termo, doc_id) por cada palavra

pares = parsed.flatMap(lambda doc: [
    (termo, doc[0])
    for termo in re.findall(r'\w+', doc[1].lower())
])

print("Primeiros 10 pares (termo, doc_id) emitidos:")
for par in pares.take(10):
    print(f"  {par}")

In [ ]:
# Passo 2 — equivalente ao Shuffle & Sort do Hadoop
# Agrupa todos os doc_id pelo mesmo termo

agrupado = pares.groupByKey()

# Neste ponto ainda é lazy — o shuffle ainda não aconteceu
# Só acontece quando chamarmos uma acção
print("RDD agrupado criado (ainda lazy, shuffle ainda não executou)")
print(f"Tipo: {type(agrupado)}")

In [ ]:
# Passo 3 — equivalente ao reducer.py
# Para cada termo, remove duplicados e ordena os doc_id

indice = agrupado.mapValues(lambda doc_ids: sorted(set(doc_ids)))

# .take(30) é a acção que despoleta tudo: leitura, flatMap, shuffle, mapValues
# Abre http://localhost:4040 agora para ver o job a executar
print("Amostra do índice invertido (20 termos, ordenados):")
for termo, doc_ids in sorted(indice.take(30), key=lambda x: x[0])[:20]:
    print(f"  {termo:25s} → {doc_ids}")

In [ ]:
# Versão compacta — as três operações encadeadas numa só expressão
# É exactamente o mesmo resultado, escrito de forma mais fluída

indice = (
    parsed
    .flatMap(lambda doc: [
        (termo, doc[0])
        for termo in re.findall(r'\w+', doc[1].lower())
    ])
    .groupByKey()
    .mapValues(lambda doc_ids: sorted(set(doc_ids)))
)

# Quantos termos distintos tem o índice?
print(f"Termos distintos no índice: {indice.count()}")

---
## 4. Guardar o Índice

Tal como no exercício MapReduce, o resultado pode ser guardado em ficheiro.
Em Spark, `.saveAsTextFile()` escreve directamente dos executores, sem passar pelo driver.

In [ ]:
import shutil, os

output_path = "output/indice_invertido"
if os.path.exists(output_path):
    shutil.rmtree(output_path)   # Spark falha se o directório já existir

indice.saveAsTextFile(output_path)
print(f"Índice guardado em {output_path}/")

# Verificar
!head -10 output/indice_invertido/part-00000

---
## 5. Experimenta

Pesquisa um termo no índice:

In [ ]:
meu_termo = "índice"

resultado = indice.filter(lambda x: x[0] == meu_termo).collect()

if resultado:
    _, doc_ids = resultado[0]
    print(f"O termo '{meu_termo}' aparece em {len(doc_ids)} documento(s): {doc_ids}")
else:
    print(f"Termo '{meu_termo}' não encontrado no índice.")

---
## 6. Terminar

In [ ]:
sc.stop()
print("SparkContext terminado.")